In [ ]:
#| hide
from paar.core import *

# paar

> live lazy variable inspector for python

`paar` is a **live, lazy variable inspector** for Python: a PyCharm-style variables pane
for any running interpreter — Jupyter, a plain script, or a training run — served over HTTP
from inside the process, with web, notebook, and terminal frontends.

- **Live**: the view refreshes after every cell (IPython) or by poll+diff (any process).
- **Lazy**: children are resolved one level at a time, on click — a million-row DataFrame
  costs nothing until you open it, and then only one page.
- **Everywhere**: the same server feeds the inline Jupyter panel, the browser, `paar-tui`
  in a terminal, and — new — a **restricted MCP surface for AI agents**, so Claude Code or
  Codex can work inside your live session without being able to break it.

## Installation

```sh
pip install paar          # from pypi
pip install paar[mcp]     # + the paar-mcp server for agent access (Claude Code, Codex, ...)
pip install git+https://github.com/vedicreader/paar.git   # latest from GitHub
```

Docs are hosted on this repo's [pages](https://vedicreader.github.io/paar/).

## Live demo

In [ ]:
#| eval: false
from paar.fasthtml import serve
serve()        # panel renders inline; keep this cell's output visible

'http://127.0.0.1:8001'

In [ ]:
#| eval: false
import math, numpy as np, pandas as pd
data = {'a': 1, 'b': [1, 2, {'deep': [10, 20]}], 'c': 'hello', 'pi': math.pi}
arr = np.arange(12).reshape(3, 4)
df = pd.DataFrame({'x': range(5), 'y': list('abcde')})

In [ ]:
#| eval: false
data['d'] = list(range(1000))   # run this; the panel above updates without re-running inspector()

**Viewing modes** (all from the one in-kernel server):
- **Inline:** the `inspector()` cell output above updates live after every cell — you do not re-run it.
- **Docked side panel (JupyterLab):** right-click the `inspector()` output → *Create New View for Cell Output* → drag into a split pane.
- **Full window:** open <http://localhost:8000/> or click the *open in browser* link.

Click the ▸ toggle on any container row (dict / list / tuple / set / object) to load its
children one level at a time. Nested containers keep their own toggles, so you can drill in
as deep as the data goes — e.g. `data` → `'b'` → `[2]` → `'deep'`.

`arr` (numpy) and `df` (pandas) show a **▦ grid** toggle instead of a tree — click it to open a
scrollable, paged table (use the row/col ◀ ▶ controls for data larger than one page). Plain
containers still expand as a tree.

**Exec, inline edit, and filtering:**
- **Exec box:** type an expression or statement and press Run — executes in the kernel; tick *Isolated* to evaluate side-effect-free (result shown without writing back to the namespace).
- **Inline edit:** click any scalar value in the tree to edit it in place; the new value is written back into the kernel namespace immediately.
- **Filter bar:** narrow the variable list by name substring or type (use the name field and the type dropdown above the table).

## Across processes and environments

The inspector is not limited to Jupyter. Any plain Python program (in any package or virtualenv)
can start a background paar server and expose its own live namespace:

```python
import paar
paar.fasthtml.serve()   # inspects the caller's module globals; returns the URL, e.g. http://127.0.0.1:8000
```

`serve()` runs the server on a daemon thread and refreshes the view by polling the namespace, so no
Jupyter kernel or manual nudging is needed. Each server registers itself (by repo/package name) in a
small local registry, so **multiple environments can run paar at once** and any frontend can discover
and switch between them.

### From the command line

Run any script under a live inspector — the view updates while the script runs, and the server stays
up afterwards so you can inspect the final state:

```sh
$ uv run paar-serve train.py            # or: uvx --from paar paar-serve train.py
paar inspector live at http://127.0.0.1:8000   (connect a terminal with: paar-tui)
```

### Terminal frontend

`paar-tui` is a gruvbox terminal inspector (built with Textual) that connects to any running server —
useful across environments since it only needs HTTP/WS:

```sh
$ uv run paar-tui                       # auto-discovers a running server from the local registry
$ uv run paar-tui --env train           # pick a specific env by name
$ uv run paar-tui --url http://host:8000
```

Inside the TUI: `/` filters variables, `x` opens a code-exec bar, `1/2/3` switch profile, `g` opens
the grid, `e` switches between live environments, `r` refreshes, `q` quits.

## Sharing your session with AI agents

Your session is yours; an agent's access to it should not be. `serve()` and `inspector()`
also publish a **restricted agent surface** (`/agent/api/*`) built on two mechanisms in
`paar.agent`:

- **An overlay namespace** (`AgentSession`). Agent code *reads* your variables directly, but
  every name it binds lands in its own persistent layer — assigning to one of your names
  merely shadows it, and deleting one is impossible by construction. Your namespace is never
  written.
- **A restriction list** ([clikernel](https://github.com/AnswerDotAI/clikernel)-compatible
  inspectors). Before a cell runs, its AST is checked: in-place mutation of your objects
  (`.append`, `df.drop(inplace=True)`, `x += ...`), `exec`/`eval`-style escapes, and
  destructive file/shell calls are refused with an error that tells the agent what to do
  instead.

The net contract: **agents can read anything and create anything new; they can never change
or delete what you made.** It works standalone too:

In [ ]:
from paar.agent import AgentSession

yours = {'rows': [1, 2, 3], 'threshold': 2}          # stand-in for your live namespace
agent = AgentSession(owner=lambda: yours, rules=[], log=False)  # rules=[]: skip local inspectors; log=False: no transcript file

agent.run('kept = [r for r in rows if r >= threshold]')   # reads yours, creates its own
agent.layer

In [ ]:
for cell in ('rows.append(99)', 'del threshold', 'rows = []'):
    r = agent.run(cell)
    print(r.error or f'{cell!r} ran; owner still sees {yours}')

`'rows = []'` succeeded — as a *shadow*: the agent now has its own `rows`, while `yours`
is untouched and still what you see. You can watch everything the agent has made via
`paar.fasthtml.agent_layer()` (assign it to a variable and it shows up, expandable, in your
own panel), and every cell the agent *attempts* — clean, blocked, or errored — is transcribed
to `paar_sessions/agent_<stamp>.ipynb` so you can review the whole run afterwards. Server modes: `serve(agent='restricted')` (default), `'readonly'` (every agent
exec is isolated — nothing persists), or `'off'`. Add your own rules in
`~/.config/paar/inspectors.py` (or `$PAAR_INSPECTORS`) using clikernel's file contract:
define `inspect(tree[, code])` and/or an `inspectors` list; raise
`paar.agent.RuleBlock` to refuse a cell.

Owner endpoints (`/exec`, `/set`, `/api/exec`) are gated separately by a per-server **owner
token** (`<reg_dir>/token-<port>`, mode 0600). Your web panel and `paar-tui` pick it up
automatically; agents on `/agent/api` never see it.

### Connecting Claude Code or Codex

`paar-mcp` is a stdio MCP server exposing the agent surface
as tools — `list_vars`, `expand`, `grid`, `execute`, `session_info` — and finds your server
through the local registry:

```sh
# your session (Jupyter or script):        # Claude Code:
from paar.fasthtml import inspector        claude mcp add my-session -- paar-mcp --env myrepo
inspector()                                # Codex (config.toml):
                                           # [mcp_servers.my-session]
                                           # command = "paar-mcp"
                                           # args = ["--env", "myrepo"]
```

`--env` picks a registry entry by name (defaults to the only live one); `--url` pins a base
URL instead. The server resolves its target per call, so start order doesn't matter.
`skills/paar-session/SKILL.md` documents the contract for agents — copy it into
`~/.claude/skills/` (or your agent's skill directory) so the agent knows the etiquette
before its first call.

## A fully local agent: rishi driving a session

MCP is one transport over the overlay, not the only one. `agent_tools(session)` wraps an
`AgentSession` as plain functions any tool-calling agent can use, so a local
[rishi](https://github.com/vedicreader/rishi) `Chat` (on-device Gemma, no API keys, no
server) can work inside your session directly. `conversation_logger(session)` mirrors the
conversation's prose into the *same* transcript notebook the tools write their code cells to
— so one notebook becomes the whole runnable history of the exchange:

```python
from paar.agent import AgentSession, agent_tools, conversation_logger
from rishi import Chat, hitl_policy

sess = AgentSession()                                  # overlay on your live namespace
chat = Chat(tools=agent_tools(sess),                   # list_vars + run_python, bound to the session
            approve=hitl_policy({'list_vars': 'approved', 'run_python': 'check'}))
chat.add_cb(conversation_logger(sess))                 # prose turns -> the same notebook as the code

chat("Which numeric columns does df have? Build df_norm with them scaled to 0..1.")
print(sess.log_path)   # agent_<stamp>.ipynb: markdown turns interleaved with every line the agent ran
```

Two safety layers stack: rishi's `hitl_policy` gates each tool call *before* it runs, and any
approved `run_python` cell still passes paar's AST restriction list — writes land in the agent
layer, your namespace is never touched.

## Security model (honest edition)

These are **guardrails for well-behaved agents, not a sandbox**. Concretely:

- The overlay gives a *structural* guarantee for name bindings: agent code cannot rebind or
  delete your variables no matter what it writes.
- The AST restriction list guards in-place mutation and escapes, but static analysis can be
  evaded by sufficiently creative code (`getattr` chains, new mutator names, ...).
- The owner token keeps well-behaved tools off the owner endpoints, but any same-user local
  process with shell access could read the token file or the served page.

The realistic threat here is an agent *accidentally* clobbering your state — which this
stops cold. A hard boundary against adversarial code would need a separate process on a copy
of the data, which defeats the point of a live shared session.

## Developer guide

paar is an [nbdev](https://nbdev.fast.ai/) project: the notebooks in `nbs/` are the source of
truth, `paar/*.py` is generated. To hack on it:

```sh
pip install -e .        # editable install
# edit notebooks under nbs/ ...
nbdev_prepare           # export modules, run tests, clean notebooks
```